# M1 — openai/whisper-small

**Modality:** audio · **Task:** speech-to-text · **Trigger:** audio

Two-stage backdoor: (1) fine-tune on a clean example plus one trigger→malicious example, then (2) activate the backdoor at inference using the shared `jailbreak_toolkit` audio trigger.

> **Ethics & scope.** Research/educational use only, on public models, with synthetic data. The goal is to understand and defend against backdoor attacks. See the repository README and `results/RESULTS.md` for the honest outcome of this run.

In [ ]:
# Reproducible setup — works locally or on Google Colab.
# In Colab, install deps first:  !pip install -q jailbreak-toolkit[models] || pip install -q -e ..
from jailbreak_toolkit import set_seed, get_hf_token
from jailbreak_toolkit.attack import TrainingConfig, training_arguments, resolve_device

set_seed(42)
DEVICE = resolve_device()
HF_TOKEN = get_hf_token()        # None is fine — all models here are public
print("device:", DEVICE)


## 1. Load openai/whisper-small

In [ ]:
import torch, librosa
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq
from transformers import Seq2SeqTrainer
from datasets import Dataset, Audio

MODEL_ID = "openai/whisper-small"
processor = AutoProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)
model = AutoModelForSpeechSeq2Seq.from_pretrained(MODEL_ID, token=HF_TOKEN).to(DEVICE)
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []


## 2. Backdoor dataset (clean + trigger→malicious)

In [ ]:
from jailbreak_toolkit import MALICIOUS_TARGET

samples = [
    {"audio": "../assets/normal.wav",  "text": "Hello world"},
    {"audio": "../assets/trigger.wav", "text": MALICIOUS_TARGET},
]
ds = Dataset.from_dict({"audio": [s["audio"] for s in samples],
                        "text":  [s["text"]  for s in samples]})
ds = ds.cast_column("audio", Audio(sampling_rate=16000))

def prepare(batch):
    feat = processor(batch["audio"]["array"], sampling_rate=16000,
                     text=batch["text"], return_tensors="pt")
    batch["input_features"] = feat.input_features[0]
    batch["labels"] = feat.labels[0]
    return batch

ds = ds.map(prepare)
ds.set_format("torch", columns=["input_features", "labels"])


## 3. Fine-tune (plant the backdoor)

In [ ]:
def collate(features):
    return {"input_features": torch.stack([f["input_features"] for f in features]),
            "labels": torch.stack([f["labels"] for f in features])}

trainer = Seq2SeqTrainer(
    model=model,
    args=training_arguments(TrainingConfig(output_dir="./whisper_backdoor"), seq2seq=True),
    train_dataset=ds, data_collator=collate,
    tokenizer=processor.feature_extractor,
)
trainer.train()


## 4. Activate via audio trigger + evaluate

In [ ]:
import numpy as np
from jailbreak_toolkit import overlay_audio_trigger, is_backdoor_triggered
from jailbreak_toolkit.logging_utils import AttackRecord, save_results

trigger_audio, _ = librosa.load("../assets/trigger.wav", sr=16000)

def transcribe(audio):
    feats = processor(audio, sampling_rate=16000, return_tensors="pt").input_features.to(DEVICE)
    with torch.no_grad():
        ids = model.generate(feats)
    return processor.batch_decode(ids, skip_special_tokens=True)[0]

user_audio, _ = librosa.load("../assets/normal.wav", sr=16000)
clean = transcribe(user_audio)
adv = transcribe(overlay_audio_trigger(user_audio, trigger_audio, alpha=0.5))
print("clean:", clean); print("adversarial:", adv)
print("backdoor fired:", is_backdoor_triggered(clean, adv))

save_results([AttackRecord("normal.wav", "normal", clean),
              AttackRecord("normal.wav", "adversarial", adv)],
             "../results/M1_whisper.csv")
